In [14]:
import numpy as np
import open3d as o3d
from plyfile import PlyData 
from pycpd import DeformableRegistration
from scipy.spatial import cKDTree


Step 1. Read both meshes in and convert to point clouds.

pixie_mesh = Mesh from PIXIE model
alpha_mesh = Mesh derived geometrically from depth data
 

In [15]:
# read in the meshes
alpha_mesh = o3d.io.read_triangle_mesh("/Users/adeleyounis/Desktop/Capstone/wAI/3D-processing/alpha_mesh.obj")
alpha_mesh.compute_vertex_normals()

pixie_mesh = o3d.io.read_triangle_mesh("/Users/adeleyounis/Desktop/Capstone/wAI/3D-processing/modelling/output/adele_down/adele_down.obj")
pixie_mesh.compute_vertex_normals()

alpha_mesh.paint_uniform_color([0.7, 0.7, 0.7])
pixie_mesh.paint_uniform_color([1.0, 0.2, 0.2])

# verify that they are triangular meshes with points and triangles
print(f"alpha_mesh: {alpha_mesh}")
print(f"pixie_mesh: {pixie_mesh}")

# o3d.visualization.draw_geometries(
#     [alpha_mesh, pixie_mesh],
#     mesh_show_back_face=True
# )

alpha_mesh: TriangleMesh with 7143 points and 14682 triangles.
pixie_mesh: TriangleMesh with 11313 points and 20908 triangles.


In [16]:
# scale and orient since pixie is scaled down
depth_size = np.array(alpha_mesh.get_max_bound() - alpha_mesh.get_min_bound())
pixie_size = np.array(pixie_mesh.get_max_bound() - pixie_mesh.get_min_bound())

scale =  pixie_size.max() / depth_size.max()
print(scale)

alpha_mesh.scale(scale, center=(0,0,0))

# Flip 180° around X (fix upside down)
pixie_mesh.rotate(pixie_mesh.get_rotation_matrix_from_xyz((np.pi, 0, 0)), center=(0,0,0))

# Rotate 180° around Y (face the camera)
pixie_mesh.rotate(pixie_mesh.get_rotation_matrix_from_xyz((0, np.pi, 0)), center=(0,0,0))

depth_center = alpha_mesh.get_center()
pixie_center = pixie_mesh.get_center()

pixie_mesh.translate(depth_center - pixie_center)

0.3075082451255584


TriangleMesh with 11313 points and 20908 triangles.

In [17]:
# get head from pixie model - delete alpha model head
min_bound = alpha_mesh.get_min_bound()
max_bound = alpha_mesh.get_max_bound()

print(min_bound, max_bound)

verts = np.asarray(alpha_mesh.vertices)
y_min, y_max = min_bound[1], max_bound[1]
x_min, x_max = min_bound[0], max_bound[0]
x_mean = (x_min + x_max) / 2

# Depth arms = points far left or far right
# (tighter threshold = fewer points kept)
arm_threshold = 0.25 * (x_max - x_min)

left_arm_mask  = np.asarray(alpha_mesh.vertices)[:,0] < (x_mean - arm_threshold)
right_arm_mask = np.asarray(alpha_mesh.vertices)[:,0] > (x_mean + arm_threshold)

depth_arm_mask = left_arm_mask | right_arm_mask
head_mask = verts[:,1] > (y_min + 0.850 * (y_max - y_min))  # top 15%


[-0.21167422 -0.71702617  2.33281898] [0.26889874 0.79857124 3.07880337]


In [18]:
# convert to point cloud
alpha_pcd = alpha_mesh.sample_points_uniformly(number_of_points=50000)
pixie_pcd = pixie_mesh.sample_points_uniformly(number_of_points=50000)

alpha_pcd.estimate_normals()
pixie_pcd.estimate_normals()

print(len(alpha_pcd.points))
print(len(pixie_pcd.points))

50000
50000


In [ ]:
# global rigid registration with RANSAC
# RANSAC samples downsampled point clouds and uses FPFH (fast point feature histograms) features
def preprocess(pcd, voxel=0.02):
    p = pcd.voxel_down_sample(voxel)
    p.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=voxel*2, max_nn=30))
    f = o3d.pipelines.registration.compute_fpfh_feature(
        p, o3d.geometry.KDTreeSearchParamHybrid(radius=voxel*5, max_nn=100)
    )
    return p, f

alpha_pcd = alpha_mesh.sample_points_poisson_disk(50000)
pixie_pcd = pixie_mesh.sample_points_poisson_disk(50000)

source_down, source_f = preprocess(pixie_pcd, voxel=0.03)
target_down, target_f = preprocess(alpha_pcd, voxel=0.03)

result_ransac = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source_down, target_down, source_f, target_f,
    mutual_filter=True,
    max_correspondence_distance=0.03*1.5, 
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
    ransac_n=4,
    checkers=[
        o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
        o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(0.03*1.5),
    ],
    criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(50000, 1000)
)

# refine with point-to-plane ICP after RANSAC 
pixie_pcd.estimate_normals()
alpha_pcd.estimate_normals()

result_icp = o3d.pipelines.registration.registration_icp(
    pixie_pcd, alpha_pcd,
    max_correspondence_distance=0.03*2.5,
    init=result_ransac.transformation,
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPlane(),
    criteria=o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=200)
)

pixie_mesh.transform(result_icp.transformation)
o3d.visualization.draw_geometries(
    [alpha_mesh, pixie_mesh],
    mesh_show_back_face=True
)

# iteration 2 - for leg alignment
def extract_legs(mesh):
    V = np.asarray(mesh.vertices)
    y_min, y_max = V[:,1].min(), V[:,1].max()
    knee_cut = y_min + 0.45 * (y_max - y_min)

    leg_mask = V[:,1] < knee_cut
    return mesh.select_by_index(np.where(leg_mask)[0], cleanup=True)

pixie_legs = extract_legs(pixie_mesh)
alpha_legs = extract_legs(alpha_mesh)

pixie_legs_pcd = pixie_legs.sample_points_poisson_disk(15000)
alpha_legs_pcd = alpha_legs.sample_points_poisson_disk(15000)

pixie_legs_pcd.estimate_normals()
alpha_legs_pcd.estimate_normals()

leg_icp = o3d.pipelines.registration.registration_icp(
    pixie_legs_pcd,
    alpha_legs_pcd,
    max_correspondence_distance=0.05,
    init=np.eye(4),
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPlane(),
    criteria=o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=150)
)

T_legs = leg_icp.transformation


pixie_V = np.asarray(pixie_mesh.vertices)
pixie_leg_idx = np.where(
    np.asarray(pixie_mesh.vertices)[:,1] <
    (pixie_V[:,1].min() + 0.55 * pixie_V[:,1].ptp())
)[0]

pixie_V_leg = pixie_V[pixie_leg_idx]
pixie_V_leg_h = np.c_[pixie_V_leg, np.ones(len(pixie_V_leg))]

pixie_V_leg_warped = (T_legs @ pixie_V_leg_h.T).T[:, :3]

pixie_V[pixie_leg_idx] = pixie_V_leg_warped
pixie_mesh.vertices = o3d.utility.Vector3dVector(pixie_V)
pixie_mesh.compute_vertex_normals()

o3d.visualization.draw_geometries([alpha_legs, pixie_legs], mesh_show_back_face=True)


[Open3D WARNING] Too few correspondences (86) after mutual filter, fall back to original correspondences.
[Open3D WARNING] [RemoveDuplicatedTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDegenerateTriangles] This mesh contains triangle uvs that are not handled in this function


Step 1. Rigid Transformation:
Aligns PIXIE to depth mesh using ICP (point-to-plane)


In [ ]:
reg = o3d.pipelines.registration.registration_icp(
    source=pixie_pcd,
    target=alpha_pcd,
    max_correspondence_distance=0.3,
    init=np.eye(4),
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPlane(),
    criteria=o3d.pipelines.registration.ICPConvergenceCriteria(
        max_iteration=5000)
)

pixie_mesh.transform(reg.transformation)



TriangleMesh with 11313 points and 20908 triangles.

In [21]:
o3d.visualization.draw_geometries(
    [alpha_mesh, pixie_mesh],
    mesh_show_back_face=True
)

Step 2: Identify PIXIE mesh parts to remove


In [7]:
pixie_normals = np.asarray(pixie_mesh.vertex_normals)
pixie_vertices_full = np.asarray(pixie_mesh.vertices)

head_mask_pixie = pixie_vertices_full[:,1] > (
    pixie_vertices_full[:,1].min() + 0.85 * (pixie_vertices_full[:,1].ptp())
)

# Arms = x far from center
x_min, x_max = pixie_vertices_full[:,0].min(), pixie_vertices_full[:,0].max()
x_center = (x_min + x_max) / 2
arm_thresh = 0.25 * (x_max - x_min)

arm_mask_pixie = (pixie_vertices_full[:,0] < x_center - arm_thresh) | \
                 (pixie_vertices_full[:,0] > x_center + arm_thresh)

# Keep head + arms from PIXIE
pixie_head_arms_mask = head_mask_pixie | arm_mask_pixie
pixie_head_arms = pixie_mesh.select_by_index(
    np.where(pixie_head_arms_mask)[0].tolist(),
    cleanup=True
)

pixie_normals = np.asarray(pixie_mesh.vertex_normals)
back_mask = pixie_normals[:,2] >= -0.25   # normals NOT facing camera

pixie_back_mask = back_mask & (~pixie_head_arms_mask)
pixie_back_only = pixie_mesh.select_by_index(
    np.where(pixie_back_mask)[0].tolist(),
    cleanup=True
)

depth_verts = np.asarray(alpha_mesh.vertices)

head_mask_depth = depth_verts[:,1] > (
    depth_verts[:,1].min() + 0.80 * depth_verts[:,1].ptp()
)

x_min, x_max = depth_verts[:,0].min(), depth_verts[:,0].max()
x_center = (x_min + x_max) / 2
arm_thresh = 0.25 * (x_max - x_min)
arm_mask_depth = (depth_verts[:,0] < x_center - arm_thresh) | \
                 (depth_verts[:,0] > x_center + arm_thresh)

remove_depth_mask = head_mask_depth | arm_mask_depth
keep_depth_mask = ~remove_depth_mask

depth_filtered = alpha_mesh.select_by_index(
    np.where(keep_depth_mask)[0].tolist(),
    cleanup=True
)

alpha_pcd = depth_filtered.sample_points_poisson_disk(50000)
tree = o3d.geometry.KDTreeFlann(alpha_pcd)

pixie_back_vertices = np.asarray(pixie_back_only.vertices)
keep = []

for i, v in enumerate(pixie_back_vertices):
    _, idx, dist = tree.search_knn_vector_3d(v, 1)
    if dist[0] > 0.0001:
        keep.append(i)

pixie_back_pruned = pixie_back_only.select_by_index(keep, cleanup=True)

combined = pixie_head_arms + pixie_back_pruned + depth_filtered
combined.remove_duplicated_vertices()
combined.remove_duplicated_triangles()
combined.remove_non_manifold_edges()
combined.compute_vertex_normals()

o3d.visualization.draw_geometries(
    [combined],
    mesh_show_back_face=True
)


[Open3D WARNING] [RemoveDuplicatedTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDegenerateTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDuplicatedTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDegenerateTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDuplicatedTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDegenerateTriangles] This mesh contains triangle uvs that are not handled in this function


Non-rigid transformations

In [23]:
# read in the meshes
alpha_mesh = o3d.io.read_triangle_mesh("/Users/adeleyounis/Desktop/Capstone/wAI/3D-processing/alpha_mesh.obj")
alpha_mesh.compute_vertex_normals()

pixie_mesh = o3d.io.read_triangle_mesh("/Users/adeleyounis/Desktop/Capstone/wAI/3D-processing/modelling/output/adele_down/adele_down.obj")
pixie_mesh.compute_vertex_normals()

alpha_mesh.paint_uniform_color([0.7, 0.7, 0.7])
pixie_mesh.paint_uniform_color([1.0, 0.2, 0.2])

# verify that they are triangular meshes with points and triangles
print(f"alpha_mesh: {alpha_mesh}")
print(f"pixie_mesh: {pixie_mesh}")

alpha_mesh: TriangleMesh with 7143 points and 14682 triangles.
pixie_mesh: TriangleMesh with 11313 points and 20908 triangles.


In [ ]:
def mesh_to_points(mesh, n=20000):
    pcd = mesh.sample_points_uniformly(number_of_points=n)
    return np.asarray(pcd.points)

# 1) sample points (use fewer first to tune)
X = mesh_to_points(pixie_mesh, n=15000)  # source (will deform)
Y = mesh_to_points(alpha_mesh, n=15000)  # target

# 2) run CPD non-rigid
reg = DeformableRegistration(X=X, Y=Y, beta=2.0, lamb=3.0, max_iterations=80, tol=1e-5)
TY, _ = reg.register()  # TY is X deformed toward Y (same shape as X)

# 3) warp the SOURCE mesh vertices using the CPD deformation learned on sampled points
V = np.asarray(alpha_mesh.vertices)
tree = cKDTree(X)
_, idx = tree.query(V, k=1)
V_warped = TY[idx]

alpha_warped = o3d.geometry.TriangleMesh(
    vertices=o3d.utility.Vector3dVector(V_warped),
    triangles=alpha_mesh.triangles
)
alpha_warped.compute_vertex_normals()
alpha_warped.paint_uniform_color([0.2, 0.8, 0.2])

o3d.visualization.draw_geometries([pixie_mesh, alpha_warped])


KeyboardInterrupt: 

: 